In [1]:
import os
import sys
from pathlib import Path
import logging
import torch
import gc

# =====================================================================
# 1. SETUP ĐƯỜNG DẪN CHỐNG LỖI (Robust Path)
# =====================================================================
# Thay vì dùng os.getcwd(), dùng đường dẫn tuyệt đối của file hiện tại
# Dành cho Jupyter Notebook:
root_dir = Path(os.path.abspath('')).parent 
# (Nếu lưu dưới dạng file .py, hãy dùng: root_dir = Path(__file__).resolve().parent.parent)

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

from src.config import TrainConfig
from notebooks.iterative_driver_voting import run_experiment_for_setting

# =====================================================================
# 2. KHAI BÁO DANH SÁCH TREO MÁY
# =====================================================================
noise_types_to_run = [
    # "worse_label", 
    # "aggre_label", 
    "random_label1", 
    # "random_label2", 
    # "random_label3"
]

# ls_alpha = [0.5, 0.8, 0.3]
# ls_alpha = [ 0.8, 0.3]
ls_alpha = [ 0.8]
# ls_alpha = [ 0.3]

logger.info(f"🚀 TIẾN TRÌNH TREO MÁY BẮT ĐẦU: {len(noise_types_to_run) * len(ls_alpha)} thực nghiệm.")
logger.info(f"📁 Thư mục gốc dự án: {root_dir}")

# =====================================================================
# 3. VÒNG LẶP TREO MÁY (SWEEP LOOP)
# =====================================================================
for alpha_val in ls_alpha:
    for current_noise in noise_types_to_run:
        logger.info("\n" + "="*70)
        logger.info(f"🔥 ĐANG CHẠY: NHIỄU = {current_noise.upper()} | ALPHA = {alpha_val} | MODE = EMA_HARD")
        logger.info("="*70)

        # Khởi tạo mới hoàn toàn để không bị rác (leak state)
        config = TrainConfig()

        # --- CẤU HÌNH ĐƯỜNG DẪN TUYỆT ĐỐI CHỐNG LỖI ---
        # config.data_dir = str(root_dir / "data_cifar10N")
        config.data_dir = str(root_dir / "notebooks" / "data_cifar10N")
        config.output_dir = str(root_dir / "outputs") # Ép ra output chuẩn xác
        
        # --- CẤU HÌNH THÔNG SỐ CHÍNH ---
        config.exp_name = "CIFAR10N_Sweep_Auto"
        config.num_classes = 10
        config.img_size = 224
        
        # KHÓA CỨNG MODEL ARCHITECTURE
        config.pretrained = True
        config.use_model_from_hf = False 
        
        # CHỐT THUẬT TOÁN
        config.noise_ratio = 0.0 # Bypass metadata validation
        config.filter_mode = "ema_hard"
        config.alpha = alpha_val                   
        
        # config.max_iterations = 10           
        # config.max_epochs_per_iter = 50      
        
        config.device = "cuda" if torch.cuda.is_available() else "cpu"
        config.use_amp = torch.cuda.is_available()

        # --- ĐỊNH VỊ DỮ LIỆU ---
        csv_dir = Path(config.data_dir) / "csvs" / f"noise_{current_noise}"
        train_csv = str(csv_dir / "train.csv")
        val_csv = str(csv_dir / "val.csv")
        test_csv = str(csv_dir / "test.csv")

        if not os.path.exists(train_csv):
            logger.warning(f"⚠️ Bỏ qua {current_noise}: Không tìm thấy file {train_csv}.")
            continue

        # --- TẠO FOLDER OUTPUT ĐỘC LẬP ---
        config.exp_dir = str(Path(config.output_dir) / config.exp_name / f"noise_{current_noise}" / f"alpha_{alpha_val}" / f"mode_{config.filter_mode}")
        os.makedirs(config.exp_dir, exist_ok=True)

        # --- THỰC THI & QUẢN LÝ TÀI NGUYÊN (BULLETPROOF) ---
        try:
            run_experiment_for_setting(config, train_csv, val_csv, test_csv)
            logger.info(f"✅ HOÀN TẤT THÀNH CÔNG: {current_noise} | Alpha={alpha_val}")
            
        except Exception as e:
            logger.error(f"❌ XẢY RA LỖI KHI CHẠY {current_noise} (Alpha={alpha_val}): {e}")
            logger.info("⏭️ Bỏ qua và chuyển sang cấu hình tiếp theo...")
            
        finally:
            # Thu gom rác hệ thống và GPU để tránh Out Of Memory (OOM)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                logger.info("🧹 Đã làm sạch CUDA VRAM, sẵn sàng cho vòng lặp mới.")

logger.info("\n🎉🎉🎉 ĐÃ HOÀN TẤT TOÀN BỘ CHIẾN DỊCH TREO MÁY! 🎉🎉🎉")

2026-05-14 06:26:28,157 - 🚀 TIẾN TRÌNH TREO MÁY BẮT ĐẦU: 1 thực nghiệm.
2026-05-14 06:26:28,157 - 📁 Thư mục gốc dự án: /mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project
2026-05-14 06:26:28,158 - 
2026-05-14 06:26:28,158 - 🔥 ĐANG CHẠY: NHIỄU = RANDOM_LABEL1 | ALPHA = 0.8 | MODE = EMA_HARD
2026-05-14 06:26:28,158 - ======================================================================
2026-05-14 06:26:28,333 - === Iteration 0 | filter_mode=ema_hard | alpha=0.8000 | noise=0.00 ===
2026-05-14 06:26:28,553 - Train samples used (iter 0): 45000/45000 (train_label_col=label_noisy)
2026-05-14 06:26:28,577 - Initializing ResNet18 for all dataset (ImageNet1K Pretrained)...
2026-05-14 06:26:28,928 - Loaded ResNet18 with 10 output classes on device cuda
/mnt/c/Users/truon/learning/ptit/research/trung/M_10_01_2025/code_v2/project/src/train_loop.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` inste

In [2]:
c

NameError: name 'c' is not defined